In [1]:
import os
import torch

gpu_list = [0]
gpu_list_str = ','.join(map(str, gpu_list))
os.environ.setdefault("CUDA_VISIBLE_DEVICES", gpu_list_str)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [2]:
from torch.nn import Linear
from model.layers import Add, Clone
from model.ViT import Attention
import torch.nn.functional as F
from einops import rearrange
import torch.nn as nn
import torchvision.models as models
from torch_geometric.nn import GATv2Conv, LayerNorm
from model.ViT import Mlp, VisionTransformer


class GraphNN(nn.Module):
    def __init__(self, cell_dim=80, vit_depth=3, proto=False, ensemble=False):
        super(GraphNN, self).__init__()
        self.resnet18 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        self.resnet18 = torch.nn.Sequential(*list(self.resnet18.children())[:-1])
        
        self.embed_dim = 32 * 8
        self.head = 8
        self.dropout = 0.3
        
        self.conv1 = GATv2Conv(in_channels=512, out_channels=int(self.embed_dim/self.head), heads=self.head)
        self.norm1 = LayerNorm(in_channels=self.embed_dim)
        
        self.cell_transformer = VisionTransformer(num_classes=cell_dim, embed_dim=self.embed_dim, depth=vit_depth,
                                                  mlp_head=True, drop_rate=self.dropout, attn_drop_rate=self.dropout)
        self.proto = proto
        if self.proto:
            self.cell_proto = nn.Parameter(torch.zeros(1, 150, self.embed_dim))
            self.cell_qkv = Linear(self.embed_dim, self.embed_dim*2)
            self.cell_att = Attention(dim=self.embed_dim, num_heads=self.head, attn_drop=self.dropout, proj_drop=self.dropout)
            self.add2 = Add()
            self.clone2 = Clone()
            self.task_norm_3 = LayerNorm(self.embed_dim, eps=1e-6)
            self.task_norm_4 = LayerNorm(self.embed_dim, eps=1e-6)
            self.cell_att_W = Linear(self.embed_dim, self.embed_dim)
            torch.nn.init.xavier_uniform_(self.cell_proto)
            
        self.ensemble = ensemble
        if self.ensemble:
            self.spot_fc = Linear(in_features=512, out_features=256)
            self.spot_head = Mlp(in_features=256, hidden_features=512*2, out_features=cell_dim)
            self.local_head = Mlp(in_features=256, hidden_features=512*2, out_features=cell_dim)
            self.fused_head = Mlp(in_features=256, hidden_features=512*2, out_features=cell_dim)
        
    def forward(self, x, edge_index):
        x_spot = self.resnet18(x)
        x_spot = x_spot.squeeze()
        
        x_local = self.conv1(x=x_spot, edge_index=edge_index)
        x_local = self.norm1(x_local)
        
        x_local = x_local.unsqueeze(0)
        
        if self.proto:
            x_cell1, x_cell2 = self.clone2(x_local, 2)
            x_cell_qkv = self.cell_qkv(self.cell_proto)
            x_cell_k, x_cell_v = rearrange(x_cell_qkv, 'b n (qkv h d) -> qkv b h n d', qkv=2, h=8)
            x_cell = self.add2([x_cell1, self.cell_att_W(self.cell_att(x=x_cell2, out_k=x_cell_k, out_v=x_cell_v))])
            x_cell = self.task_norm_4(x_cell)
            x_cell = self.task_norm_3(x_cell + F.relu(x_cell))
        else:
            x_cell = x_local
        
        if self.ensemble:
            x_spot = self.spot_fc(x_spot)
            cell_predication_spot = self.spot_head(x_spot)
            x_local = x_local.squeeze(0)
            cell_prediction_local = self.local_head(x_local)
            cell_prediction_global, x_global = self.cell_transformer(x_cell)
            cell_prediction_global = cell_prediction_global.squeeze()
            x_global = x_global.squeeze()
            cell_prediction_fused = self.fused_head((x_spot+x_local+x_global)/3.0)
            # cell_prediction = (cell_predication_spot + cell_prediction_local + cell_prediction_global + cell_prediction_fused) / 4.0
            cell_prediction = (cell_predication_spot + cell_prediction_local*3.0 + cell_prediction_global + cell_prediction_fused) / 6.0
        else:  
            cell_prediction, _ = self.cell_transformer(x_cell)
            cell_prediction = cell_prediction.squeeze()
        
        # cell_prediction = torch.relu(cell_prediction)
        
        return cell_prediction

/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from glob import glob
test_slides = glob('/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/brca/*.pt')
test_slides.sort()
test_slides

['/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/brca/TCGA-3C-AALK-01Z-00-DX1.pt',
 '/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/brca/TCGA-4H-AAAK-01Z-00-DX1.pt',
 '/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/brca/TCGA-5L-AAT0-01Z-00-DX1.pt',
 '/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/brca/TCGA-5L-AAT1-01Z-00-DX1.pt',
 '/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/brca/TCGA-5T-A9QA-01Z-00-DX1.pt',
 '/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/brca/TCGA-A1-A0SB-01Z-00-DX1.pt',
 '/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/brca/TCGA-A1-A0SD-01Z-00-DX1.pt',
 '/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/brca/TCGA-A1-A0SF-01Z-00-DX1.pt',
 '/data1/r20user3/shared_project/Hist2Cell/code/data_preprocessing/rawimg_graph/brca/TCG

In [4]:
len(test_slides)

826

In [17]:
test_slide_id_list = [os.path.basename(x).split('.')[0] for x in test_slides]
test_slide_id_list[:5]

['TCGA-3C-AALK-01Z-00-DX1',
 'TCGA-4H-AAAK-01Z-00-DX1',
 'TCGA-5L-AAT0-01Z-00-DX1',
 'TCGA-5L-AAT1-01Z-00-DX1',
 'TCGA-5T-A9QA-01Z-00-DX1']

In [21]:
from glob import glob
import os

skip_slide_list = []
for test_slide_id in test_slide_id_list:
    patch_no = glob(os.path.join('/data1/r20user3/shared_project/Hist2Cell/data/brca', test_slide_id, 'patches', '*'))
    if len(patch_no) < 160:
        skip_slide_list.append(test_slide_id)

print(len(skip_slide_list))        
skip_slide_list[:5]

241


['TCGA-5L-AAT0-01Z-00-DX1',
 'TCGA-A1-A0SB-01Z-00-DX1',
 'TCGA-A1-A0SD-01Z-00-DX1',
 'TCGA-A1-A0SF-01Z-00-DX1',
 'TCGA-A1-A0SH-01Z-00-DX1']

In [25]:
from torch_geometric.data import Batch
from torch_geometric.loader import NeighborLoader
import torch_geometric
torch_geometric.typing.WITH_PYG_LIB = False
from tqdm import tqdm
import numpy as np
from scipy.stats import pearsonr
import joblib
    
ensemble = True
vit_depth = 1
celltype_num = 39
model = GraphNN(cell_dim=celltype_num, vit_depth=1, ensemble=ensemble)
checkpoint = torch.load("/data1/r20user3/shared_project/Hist2Cell/code/training/model_weights/breast_cross_source_epoch100_lr1e-4_2hop_ensemble_Trans1layer_GNNoutput50_onlycell_best_cell_all_abundance_average.pth")
model.load_state_dict(checkpoint)
model = model.to(device)    

for case in tqdm(test_slides):
    # print(case)
    case_id = case.split('/')[-1].split('.')[0]
    if case_id in skip_slide_list:
        continue
    
    test_dataset = torch.load(case)
    
    hop = 2
    subgraph_bs = 16    

    test_loader = NeighborLoader(
        test_dataset,
        num_neighbors=[-1]*hop,
        batch_size=subgraph_bs,
        directed=False,
        input_nodes=None,
        shuffle=False,
        # num_workers=8,
        # pin_memory=True, 
        # prefetch_factor=2,
    )

    with torch.no_grad():
        model.eval()

        test_sample_num = 0
        test_cell_pred_array = []
        test_cell_pos_array = []
        
        for graph in test_loader:
            x = graph.x.to(device)
            
            if x.shape[0] == 1:
                continue
            
            pos = graph.pos.to(device)
            edge_index = graph.edge_index.to(device)
            cell_pred = model(x=x, edge_index=edge_index)
                
            center_num = len(graph.input_id)
            center_cell_pred = cell_pred[:center_num, :]
            center_cell_pos = pos[:center_num, :]
            
            test_cell_pred_array.append(center_cell_pred.squeeze().cpu().detach().numpy())
            test_cell_pos_array.append(center_cell_pos.squeeze().cpu().detach().numpy())
            test_sample_num = test_sample_num + center_num
        
        
        if len(test_cell_pred_array[-1].shape) == 1:
            test_cell_pred_array[-1] = np.expand_dims(test_cell_pred_array[-1], axis=0)
        test_cell_pred_array = np.concatenate(test_cell_pred_array)
        if len(test_cell_pos_array[-1].shape) == 1:
            test_cell_pos_array[-1] = np.expand_dims(test_cell_pos_array[-1], axis=0)
        test_cell_pos_array = np.concatenate(test_cell_pos_array)
        
        
        Predictions = {
            'cell_abundance_predictions': test_cell_pred_array,
            'coords': test_cell_pos_array,
        }
        save_path = "/data1/r20user3/shared_project/Hist2Cell/code/analysis/inference/brca/breast_cross_source_epoch100_lr1e-4_2hop_ensemble_Trans1layer_GNNoutput50_onlycell_"+case_id+"_best_cell_all_abundance_average.pkl"
        joblib.dump(Predictions, save_path)

  0%|          | 0/826 [00:00<?, ?it/s]/home/r20user3/anaconda3/envs/gene/lib/python3.11/site-packages/torch_geometric/sampler/neighbor_sampler.py:50: UserWarning: Using '{self.__class__.__name__}' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn("Using '{self.__class__.__name__}' without a "
100%|██████████| 826/826 [2:01:27<00:00,  8.82s/it]  


In [24]:
x.shape[0]

1

In [10]:
xx = torch.concat([x, x], dim=0)
xx.shape

torch.Size([2, 3, 224, 224])

'TCGA-A1-A0SB-01Z-00-DX1'

In [9]:
results = joblib.load("/data1/r20user3/shared_project/Hist2Cell/code/analysis/inference/brca/breast_cross_source_epoch100_lr1e-4_2hop_ensemble_Trans1layer_GNNoutput50_onlycell_TCGA-A2-A0CS-01Z-00-DX1_best_cell_all_abundance_average.pkl")
results

{'cell_abundance_predictions': array([[0.14408234, 0.19259194, 0.07428569, ..., 0.08687232, 0.5333519 ,
         0.06829645],
        [0.13472098, 0.17712955, 0.07649646, ..., 0.0847715 , 0.7386719 ,
         0.08661173],
        [0.09150679, 0.1183451 , 0.05797069, ..., 0.04864913, 0.6514593 ,
         0.10481694],
        ...,
        [0.11019372, 0.15198171, 0.0567168 , ..., 0.06901357, 0.75038743,
         0.06381158],
        [0.09472549, 0.12652582, 0.06865953, ..., 0.05143038, 0.53732216,
         0.09230685],
        [0.14978442, 0.20097105, 0.0761504 , ..., 0.09519322, 0.40691072,
         0.07789738]], dtype=float32),
 'coords': array([[67., 65.],
        [23., 68.],
        [13., 29.],
        ...,
        [30., 71.],
        [73., 56.],
        [59., 14.]], dtype=float32)}

In [10]:
results['cell_abundance_predictions'].shape

(2754, 39)

In [12]:
results['cell_abundance_predictions'].max()

13.929008

In [13]:
results['coords']

array([[67., 65.],
       [23., 68.],
       [13., 29.],
       ...,
       [30., 71.],
       [73., 56.],
       [59., 14.]], dtype=float32)